# Tarea 19 · Tasa de baja por contrato y método de pago

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/feat/19-barras-churn/notebooks/trabajo/19_barras_churn.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Julián.

Graficamos la proporción de clientes que se dieron de baja por tipo de contrato y método de pago. Exportamos ambos gráficos en `reports/figures/`.

## Cómo ejecutarlo

- **Local:** instalá `requirements.txt`, abrí el notebook desde el repositorio y ejecutá todas las celdas.
- **Colab:** abrí el botón tras publicar la rama o subí este `.ipynb` y ejecutá todas las celdas. El CSV se lee desde GitHub y los PNG quedan en el panel **Archivos**. Al integrar la tarea, actualizar el botón a `main`.

## 1. Estilo común

Usamos la celda del punto 16 para mantener la tipografía, el color de las bajas y la exportación a 300 dpi.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

if EN_COLAB:
    RAIZ = Path.cwd()
else:
    actual = Path.cwd().resolve()
    RAIZ = next(
        (ruta for ruta in (actual, *actual.parents) if (ruta / 'PLAN_TAREAS.md').is_file()),
        None,
    )
    if RAIZ is None:
        raise FileNotFoundError('Abrí Jupyter dentro del repositorio del TPI.')

TAMANO_FIGURA = (9, 5)
DPI_EXPORTACION = 300
PALETA_CATEGORICA = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']
COLORES_CHURN = {'No': '#0072B2', 'Yes': '#D55E00'}
MAPA_CORRELACION = 'vlag'
CARPETA_FIGURAS = RAIZ / 'reports' / 'figures'

def aplicar_estilo():
    """Aplica el estilo común del TPI a las figuras que se creen después."""
    sns.set_theme(
        style='whitegrid',
        context='notebook',
        font='DejaVu Sans',
        palette=PALETA_CATEGORICA,
        rc={
            'figure.figsize': TAMANO_FIGURA,
            'figure.dpi': 110,
            'figure.facecolor': 'white',
            'axes.facecolor': 'white',
            'axes.titlesize': 14,
            'axes.titleweight': 'bold',
            'axes.labelsize': 12,
            'xtick.labelsize': 11,
            'ytick.labelsize': 11,
            'legend.fontsize': 11,
            'axes.spines.top': False,
            'axes.spines.right': False,
            'grid.alpha': 0.25,
            'savefig.dpi': DPI_EXPORTACION,
            'savefig.facecolor': 'white',
        },
    )

def guardar_figura(fig, nombre):
    """Guarda fig en reports/figures/<nombre>.png a 300 dpi y devuelve Path.

    Usar un nombre sin ruta ni extensión, como '17_histograma_tenure'.
    Un nombre existente se sobrescribe para permitir regenerar el gráfico.
    Guardar antes de plt.show() y cerrar luego con plt.close(fig).
    """
    if not nombre or Path(nombre).name != nombre or Path(nombre).suffix:
        raise ValueError('Usá un nombre sin ruta ni extensión.')
    CARPETA_FIGURAS.mkdir(parents=True, exist_ok=True)
    ruta = CARPETA_FIGURAS / f'{nombre}.png'
    fig.savefig(ruta, dpi=DPI_EXPORTACION, bbox_inches='tight', facecolor='white')
    return ruta

aplicar_estilo()
print('Estilo común aplicado. Exportación PNG a 300 dpi en reports/figures/.')


## 2. Carga de datos

Usamos el CSV limpio cuando está disponible y el provisorio mientras se completa la limpieza. Seleccionamos el contrato, el método de pago y `Churn`.

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

variables = ["Contract", "PaymentMethod", "Churn"]
ruta_limpia = Path("data/processed/telco_clean.csv")
ruta_provisoria = Path("data/processed/telco_provisorio.csv")

if EN_COLAB:
    url_repo = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    fuente = url_repo + ruta_limpia.as_posix()
    try:
        datos = pd.read_csv(fuente, usecols=variables)
    except HTTPError as error:
        if error.code != 404:
            raise
        fuente = url_repo + ruta_provisoria.as_posix()
        datos = pd.read_csv(fuente, usecols=variables)
else:
    actual = Path.cwd().resolve()
    raiz = next(
        (ruta for ruta in (actual, *actual.parents)
         if (ruta / ruta_limpia).is_file() or (ruta / ruta_provisoria).is_file()),
        None,
    )
    if raiz is None:
        raise FileNotFoundError("Falta el CSV de la tarea 3 o el dataset limpio de la tarea 9.")
    fuente = raiz / ruta_limpia
    if not fuente.is_file():
        fuente = raiz / ruta_provisoria
    datos = pd.read_csv(fuente, usecols=variables)

datos = datos[variables]
print(f"Fuente: {fuente}")

## 3. Cálculo y formato de las barras

En cada categoría, dividimos las bajas (`Churn = Yes`) por el total de clientes de esa categoría. Ambos gráficos usan una escala del 0 al 100 % y muestran el porcentaje de cada barra. La línea punteada indica la tasa de toda la cartera para facilitar la comparación.

In [ ]:
assert not datos.empty and datos["Churn"].isin(["Yes", "No"]).all(), "Revisá los valores de Churn."
bajas = datos["Churn"].eq("Yes")
tasa_general = bajas.mean()
nombre_fuente = str(fuente).rsplit("/", 1)[-1] if EN_COLAB else fuente.name

def graficar_tasa_baja(columna, etiquetas, titulo, nombre_archivo):
    grupos = datos[columna].map(etiquetas).fillna(datos[columna]).fillna("Sin dato")
    tasas = bajas.groupby(grupos).mean().sort_values()

    fig, ax = plt.subplots(layout="constrained")
    barras = ax.barh(tasas.index, tasas, color=COLORES_CHURN["Yes"])
    ax.bar_label(
        barras, labels=[f"{tasa:.2%}" for tasa in tasas], padding=5,
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 1},
    )
    ax.axvline(tasa_general, color="#444444", linestyle="--", label=f"Cartera: {tasa_general:.2%}")
    ax.set(xlim=(0, 1), xlabel="Porcentaje de clientes que se dieron de baja", title=titulo)
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.yaxis.grid(False)
    ax.legend(loc="lower right")
    fig.supxlabel(f"Fuente: {nombre_fuente} · n = {len(datos):,}", fontsize=9)

    ruta_figura = guardar_figura(fig, nombre_archivo)
    print(f"Figura exportada: {ruta_figura.relative_to(RAIZ).as_posix()}")
    plt.show()
    plt.close(fig)
    return tasas

## 4. Tasa de baja por contrato

Comparamos la proporción de bajas entre clientes con contratos mensuales, anuales y de dos años.

In [ ]:
tasas_contrato = graficar_tasa_baja(
    "Contract",
    {"Month-to-month": "Mensual", "One year": "Un año", "Two year": "Dos años"},
    "Tasa de baja por tipo de contrato",
    "19_churn_por_contrato",
)

Con el CSV provisorio, el contrato **mensual** registra la mayor tasa de baja (42,71 %), frente al 11,27 % en contratos de un año y al 2,83 % en los de dos años. Es el único tipo de contrato por encima de la tasa general de 26,54 %.

## 5. Tasa de baja por método de pago

Comparamos la proporción de bajas entre clientes que usan cheque electrónico, cheque por correo, transferencia automática o tarjeta de crédito automática.

In [ ]:
tasas_pago = graficar_tasa_baja(
    "PaymentMethod",
    {
        "Electronic check": "Cheque electrónico",
        "Mailed check": "Cheque por correo",
        "Bank transfer (automatic)": "Transferencia automática",
        "Credit card (automatic)": "Tarjeta de crédito automática",
    },
    "Tasa de baja por método de pago",
    "19_churn_por_metodo_de_pago",
)

Con el CSV provisorio, **cheque electrónico** tiene la mayor tasa de baja (45,29 %). Le siguen cheque por correo (19,11 %), transferencia automática (16,71 %) y tarjeta de crédito automática (15,24 %). La diferencia permite identificar un grupo con más bajas observadas. Para entenderla, también importa cómo se distribuyen los tipos de contrato y servicio entre los métodos de pago.

Revisar las interpretaciones al regenerar los gráficos con `telco_clean.csv`.